# Vector Databases :

#### - This notebook shows the implementation of different vector datastores, their pros & cons and Use cases. 

In [1]:
# This is the comman code snippet, required to implement below given vectore stores
from langchain_community.embeddings import HuggingFaceEmbeddings

def get_embeddings():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

# Documents
from langchain_core.documents import Document

docs = [
    Document(
        page_content="RAG combines retrieval and generation.",
        metadata={"source": "doc1"}
    ),
    Document(
        page_content="Vector databases store embeddings.",
        metadata={"source": "doc2"}
    ),
    Document(
        page_content="FAISS is a fast similarity search library.",
        metadata={"source": "doc3"}
    )
]

---
---

## 1. FAISS (Facebook AI Similarity Search)

🔹 What it is
- A library, not a managed DB
- Runs in-process
- Extremely fast for local or server-based search

🔹 How it works
- Uses optimized C++ algorithms
- Supports multiple indexing strategies (Flat, IVF, HNSW, PQ)
- No built-in persistence unless you explicitly save

✅ Pros
```markdown
    ✔ Very fast
    ✔ Free & open-source
    ✔ No external service needed
    ✔ Perfect for learning & local apps
    ✔ Highly customizable index types
```

❌ Cons
```markdown
    ✘ No built-in metadata filtering
    ✘ No auth / multi-tenant support
    ✘ Manual persistence
    ✘ Scaling is your responsibility
```
🎯 Best Use Cases
```python
    - Local RAG systems (like DocuChat v1)
    - Research prototypes
    - On-prem deployments
    - Single-user or single-tenant systems
    - High-performance in-memory retrieval
```

In [6]:
from langchain_community.vectorstores import FAISS

def create_faiss_db(documents):
    embeddings = get_embeddings()
    vector_db = FAISS.from_documents(
        documents=documents,
        embedding=embeddings
    )
    return vector_db

faiss_db = create_faiss_db(docs)
print(f'Vector Store created: {faiss_db}')

Vector Store created: <langchain_community.vectorstores.faiss.FAISS object at 0x000001B6E3617D90>


```python
# Retrieval Functions
def faiss_retrieve(vector_db, query, k=3):
    retriever = vector_db.as_retriever(search_kwargs={"k": k})
    return retriever.invoke(query)
```

---
---

## 2. ChromaDB

🔹 What it is

- Embedding-native database
- Built specifically for LLM apps
- Can run locally or server-based

🔹 How it works

- Stores embeddings + metadata together
- Automatic persistence
- Simple Python API

✅ Pros
```markdown
    ✔ Metadata filtering
    ✔ Persistent by default
    ✔ Easy setup
    ✔ Designed for RAG
    ✔ Active LLM community support
```

❌ Cons
```markdown
    ✘ Slower than FAISS at very large scale
    ✘ Not ideal for massive distributed systems
    ✘ Fewer advanced indexing options
```

🎯 Best Use Cases
```python
    - Production-ready RAG apps
    - Multi-document Q&A
    - Apps with metadata filters (by page, doc, user)
    - Medium-scale deployments
```

In [7]:
from langchain_community.vectorstores import Chroma

def create_chroma_db(documents, persist_dir="chroma_db"):
    embeddings = get_embeddings()
    vector_db = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory=persist_dir
    )
    return vector_db

chroma_db = create_chroma_db(docs)
print(f'Vector Store created: {chroma_db}')

Vector Store created: <langchain_community.vectorstores.chroma.Chroma object at 0x000001B6E3681A90>


In [9]:
print(chroma_db.get(include=["metadatas", "embeddings"]))

{'ids': ['220b9190-b54f-42a1-9cdc-f762cf167b45', 'b096e485-8e3c-4e79-bf0c-7bdc7a1c925b', 'e66207f0-2834-4950-af6f-5488796d5cb9'], 'embeddings': array([[-0.05594323,  0.07519938,  0.0471557 , ...,  0.00538453,
         0.00115328,  0.00307641],
       [-0.00390205, -0.08049764, -0.05386188, ..., -0.02335513,
         0.04151461,  0.00516465],
       [-0.0397495 , -0.08211172, -0.08760588, ...,  0.01735895,
         0.10820428,  0.02984412]], shape=(3, 384)), 'documents': None, 'uris': None, 'included': ['metadatas', 'embeddings'], 'data': None, 'metadatas': [{'source': 'doc1'}, {'source': 'doc2'}, {'source': 'doc3'}]}


```python
# Retrieval Functions
def chroma_retrieve(vector_db, query, k=3):
    retriever = vector_db.as_retriever(
        search_kwargs={"k": k}
    )
    return retriever.invoke(query)
```

---
---

## 3. Pinecone

🔹 What it is

- Fully managed cloud vector database
- Designed for enterprise-scale similarity search

🔹 How it works

- Serverless or pod-based architecture
- Built-in scalability, backups, auth
- Supports namespaces & filtering

✅ Pros
```markdown
    ✔ Fully managed
    ✔ Massive scalability
    ✔ Metadata filtering
    ✔ High availability
    ✔ Zero infra maintenance
```

❌ Cons
```markdown
    ✘ Paid service
    ✘ Internet dependency
    ✘ Less low-level control
    ✘ Vendor lock-in
```

🎯 Best Use Cases
```python
    - SaaS RAG products    
    - Multi-tenant apps
    - Enterprise search    
    - High QPS production workloads
```

```python
# Pinecone Vector DB Defining
from langchain_community.vectorstores import Pinecone
from pinecone import Pinecone as PineconeClient

def create_pinecone_db(documents, index_name="rag-index"):
    embeddings = get_embeddings()

    pc = PineconeClient(
        api_key=
    )

    index = pc.Index(index_name)

    vector_db = Pinecone.from_documents(
        documents=documents,
        embedding=embeddings,
        index=index
    )
    return vector_db

pinecone_db = create_pinecone_db(docs)
```

```python
# Retrieval Functions
def pinecone_retrieve(vector_db, query, k=3):
    retriever = vector_db.as_retriever(
        search_kwargs={"k": k}
    )
    return retriever.invoke(query)
```

---
---

## 4. Weaviate

🔹 What it is

- Vector DB + Graph DB hybrid
- Schema-driven
- Supports hybrid (keyword + vector) search

✅ Pros
```markdown
    ✔ Hybrid search
    ✔ Rich metadata queries
    ✔ Graph relationships
    ✔ Cloud & self-hosted
```

❌ Cons
```markdown
    ✘ Steeper learning curve
    ✘ More ops overhead
    ✘ Overkill for simple RAG
```

🎯 Use Cases
```python
    - Knowledge graphs
    - Enterprise search
    - Structured + unstructured data
```
---
---

## 5. Milvus

🔹 What it is

- High-performance distributed vector database
- Designed for massive scale

✅ Pros
```markdown
    ✔ Horizontal scaling
    ✔ Cloud-native
    ✔ Very fast at large scale
```

❌ Cons
```markdown
    ✘ Complex setup
    ✘ Needs Kubernetes
    ✘ Overkill for small projects
```

🎯 Use Cases
```python
    - Billions of vectors
    - Large AI platforms
    - Production AI infra teams
```

---
---


| Vector DB | Local | Metadata | Scale      | Cost       | Best For      |
| --------- | ----- | -------- | ---------- | ---------- | ------------- |
| FAISS     | ✅     | ❌        | Low–Medium | Free       | Prototypes    |
| Chroma    | ✅     | ✅        | Medium     | Free       | RAG apps      |
| Pinecone  | ❌     | ✅        | Very High  | Paid       | SaaS          |
| Weaviate  | ✅/❌   | ✅        | High       | Mixed      | Hybrid search |
| Milvus    | ❌     | ✅        | Very High  | Infra cost | Big data      |

---

---
```python

=> Learning / Demo → FAISS

=> Local RAG App → FAISS / Chroma

=> Production RAG → Chroma / Pinecone

=> Enterprise SaaS → Pinecone / Weaviate

=> Massive Scale → Milvus
```